과제 2의 목표는 수출 데이터 벤더의 주장, 즉 **"데이터를 가장 빠르게 제공하여 클라이언트가 가격에 반영되기 전에 시그널을 확보할 수 있다"**는 주장이 타당한지 검증하는 것입니다.

검증 방법은 저희가 정의한 **핵심 시그널 (Surprise Z-Score)**이 데이터 공개 시점 전과 후에 주가에 미치는 영향을 비교 분석하는 것입니다.

In [3]:
import pandas as pd
import numpy as np
import os
from tqdm.auto import tqdm # tqdm 임포트
tqdm.pandas() # Pandas 확장 모듈 활성화

# --- 경로 및 파일 설정 ---
ARIMA_RESULT_PATH = "../output/problem1_surprise/problem1_surprise_arima.csv"
CLOSE_PRICE_PATH = "../data/price/close.csv"
OUT_DIR = "../output/problem2_vendor"
os.makedirs(OUT_DIR, exist_ok=True)


# --- 1. 데이터 로드 및 전처리 ---
try:
    df_surprise = pd.read_csv(ARIMA_RESULT_PATH)
    df_surprise['date'] = pd.to_datetime(df_surprise['date'])
    df_surprise = df_surprise[['date', 'symbol', 'surprise_z']].dropna()
    
    df_price = pd.read_csv(CLOSE_PRICE_PATH)
    df_price = df_price.rename(columns={df_price.columns[0]: 'date'})
    df_price['date'] = pd.to_datetime(df_price['date'], format='%Y%m%d')
    
except FileNotFoundError as e:
    print(f"오류: 필요한 파일을 찾을 수 없습니다. 경로를 확인하세요: {e}")
    exit()

# 가격 데이터를 긴 형태(Long format)로 변환
df_price_long = df_price.melt(id_vars=['date'], var_name='symbol', value_name='close_price')
df_price_long = df_price_long.dropna(subset=['close_price'])

# Surprise 지표와 가격 데이터 병합
df_merged = pd.merge(df_surprise, df_price_long, on=['date', 'symbol'], how='inner')


# --- 2. 공개 전후 수익률 계산 함수 ---
df_price_long = df_price_long.sort_values(['symbol', 'date']).set_index('date')

def calculate_pre_post_returns(row):
    """주가 공개일을 기준으로 직전/직후 수익률을 계산하는 함수"""
    sym = row['symbol']
    pub_date = row['date']
    df_sym = df_price_long[df_price_long['symbol'] == sym]
    
    # 1. 공개 직전 가격 (Price Pre): 발표일 이전의 가장 가까운 종가
    try:
        price_pre_day = df_sym.index[df_sym.index < pub_date].max()
        price_pre = df_sym.loc[price_pre_day, 'close_price']
    except:
        price_pre = np.nan

    # 2. 공개 직후 1일 가격 (Price Post 1): 발표일 이후의 가장 가까운 종가
    try:
        price_post_day = df_sym.index[df_sym.index > pub_date].min()
        price_post = df_sym.loc[price_post_day, 'close_price']
    except:
        price_post = np.nan

    # 3. 공개 직후 2일 가격 (Price Post 2): 2번째로 가까운 다음 거래일 종가
    try:
        post_dates = df_sym.index[df_sym.index > pub_date]
        if len(post_dates) >= 2:
            price_post_2 = df_sym.loc[post_dates[1], 'close_price']
        else:
            price_post_2 = np.nan
    except:
        price_post_2 = np.nan
        
    # 4. 수익률 계산
    return_post_1d = (price_post / price_pre) - 1 if pd.notna(price_pre) and pd.notna(price_post) and price_pre != 0 else np.nan
    return_post_2d = (price_post_2 / price_pre) - 1 if pd.notna(price_pre) and pd.notna(price_post_2) and price_pre != 0 else np.nan

    return pd.Series([return_post_1d, return_post_2d], index=['return_post_1d', 'return_post_2d'])


# --- 3. 수익률 계산 적용 (tqdm 사용) ---
print("공개 전후 수익률 계산 시작 (진행률 표시)")
# progress_apply를 사용하여 진행률 표시줄을 활성화합니다.
df_returns = df_merged.progress_apply(calculate_pre_post_returns, axis=1)
df_analysis = pd.concat([df_merged, df_returns], axis=1)
print("수익률 계산 완료.")


# --- 4. 결과 출력 및 저장 (이하 생략) ---
# ...

/Users/masterj/Documents/GitHub/team5/StockPlay-Data-analysis/5mil/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


공개 전후 수익률 계산 시작 (진행률 표시)


100%|██████████| 10443/10443 [02:31<00:00, 68.91it/s]

수익률 계산 완료.


In [4]:
# --- 1. 최종 분석 테이블 준비 ---
# return_post_1d와 return_post_2d는 이미 df_analysis에 계산되어 추가됨.
df_final = df_analysis[['date', 'symbol', 'surprise_z', 'close_price', 'return_post_1d', 'return_post_2d']]

# --- 2. 결과 파일 저장 ---
OUT_FILE_NAME = "problem2_vendor_analysis_base.csv"
output_path = os.path.join(OUT_DIR, OUT_FILE_NAME)

try:
    df_final.to_csv(output_path, index=False)
    print(f"\n[OK] 최종 분석 데이터 저장 완료: {output_path}")
except Exception as e:
    print(f"\n[ERROR] 파일 저장 실패: {e}")
    
# --- 3. 벤더 주장 검증을 위한 요약 분석 ---

# 시그널 경계 설정 (Z-score의 평균은 0, 표준편차는 1에 가깝다고 가정)
MEAN_Z_SCORE = df_final['surprise_z'].mean() 
STD_Z_SCORE = df_final['surprise_z'].std() 

# 시그널 분류 (핵심 시그널 정의: |Z| > 2)
df_final['signal_type'] = np.select(
    [
        df_final['surprise_z'] > MEAN_Z_SCORE + 2 * STD_Z_SCORE,
        df_final['surprise_z'] < MEAN_Z_SCORE - 2 * STD_Z_SCORE
    ],
    [
        'Positive_Signal',
        'Negative_Signal'
    ],
    default='No_Signal'
)

# 핵심 요약 분석: 시그널 발생 시 평균 수익률 계산
df_summary = df_final[df_final['signal_type'] != 'No_Signal'].groupby('signal_type')[['return_post_1d', 'return_post_2d']].mean()
df_summary = df_summary.mul(100).round(2).rename(columns={'return_post_1d': 'Avg_Return_Post_1D (%)', 'return_post_2d': 'Avg_Return_Post_2D (%)'})

# --- 4. 요약 결과 출력 ---
print("\n" + "="*50)
print("벤더 퀄리티 검증 (시그널별 평균 수익률 요약)")
print("="*50)
print(df_summary.to_markdown())
print("="*50)


[OK] 최종 분석 데이터 저장 완료: ../output/problem2_vendor/problem2_vendor_analysis_base.csv

벤더 퀄리티 검증 (시그널별 평균 수익률 요약)
| signal_type     |   Avg_Return_Post_1D (%) |   Avg_Return_Post_2D (%) |
|:----------------|-------------------------:|-------------------------:|
| Negative_Signal |                     0.35 |                     0.79 |
| Positive_Signal |                     0.26 |                     0.86 |


제공해주신 결과는 벤더의 주장, 특히 **음의 시그널(Negative Signal)**의 예측력에 대해 심각한 의문을 제기합니다.

결론적으로, 이 예비 분석 결과만으로는 벤더의 주장이 타당하지 않다는 잠정적인 근거가 됩니다.

📊 벤더 퀄리티 검증 결과 해석
벤더의 주장은 **"데이터 공개 직후 시장이 서프라이즈 방향대로 반응해야 한다"**는 것입니다. 저희 분석 결과(Avg_Return_Post_1D (%))는 다음과 같습니다.

Signal Type	예상되는 주가 반응	실제 Avg Return Post 1D (%)	해석
Positive Signal	주가 상승 (양수)	+0.26% (예상대로 양수)	방향성은 일치하지만, 수익률 크기가 미미합니다.
Negative Signal	주가 하락 (음수)	+0.35% (예상과 반대)	방향성이 완전히 불일치합니다. (주가가 하락해야 하는데 오히려 상승했습니다.)

1. 부정적 시그널의 예측 실패 (핵심 문제)
Negative Signal은 수출액이 예상치보다 크게 낮았다는 뜻이므로, 공개 직후 주가는 **하락(-)**해야 유효한 시그널입니다.

그러나 실제 결과는 **+0.35%**로, 주가가 하락하지 않고 상승했습니다. 심지어 Positive Signal(+0.26%)보다 더 큰 평균 상승을 기록했습니다.

이는 시그널이 시장 방향을 전혀 예측하지 못하고 있음을 의미합니다.

2. 전체적인 방향성 부족
Positive Signal의 +0.26%는 방향성은 맞지만, 전체 평균 주가 변동성을 고려할 때 시그널로서의 강도가 매우 약할 수 있습니다.

Post 2D 수익률 역시 두 경우 모두 강한 양수(+0.79%, +0.86%)를 보여주는데, 이는 시그널과 무관하게 **시장 전체가 상승 추세(Market Drift)**에 있었을 가능성 또는 시그널 후에도 일관된 반응이 없었음을 시사합니다.

💡 최종 결론 및 다음 분석
이 결과만 보면 **"ARIMA Z-Score 기반의 Negative Signal은 거짓 시그널(False Signal)이며, 벤더의 주장은 현재 분석 시점까지는 검증되지 않았다"**고 보고할 수 있습니다.

In [ ]:
import pandas as pd
import numpy as np
import os
from tqdm.auto import tqdm
tqdm.pandas()

# --- 경로 설정 ---
ARIMA_RESULT_PATH = "../output/problem1_surprise/problem1_surprise_arima.csv"
PRICE_MINUTELY_DIR = "../data/price_minutely/"
OUT_DIR = "../output/problem2_vendor"
os.makedirs(OUT_DIR, exist_ok=True)


# --- 1. 분 단위 가격 데이터 로드 및 통합 ---
print("➡️ 1. 분 단위 가격 데이터 통합 로드 시작...")

all_minutely_files = [f for f in os.listdir(PRICE_MINUTELY_DIR) if f.startswith('close_') and f.endswith('.csv')]
df_minutely_list = []

# 모든 시간대별 종가 파일을 순차적으로 로드
for file_name in all_minutely_files:
    file_path = os.path.join(PRICE_MINUTELY_DIR, file_name)
    try:
        df_temp = pd.read_csv(file_path)
        
        # 첫 번째 컬럼(날짜/시간) 이름을 'datetime'으로 지정
        df_temp = df_temp.rename(columns={df_temp.columns[0]: 'datetime'})
        
        # 파일 이름에서 시간대 추출 (예: 'close_0900_0910.csv' -> '0910')
        end_time_str = file_name.split('_')[-1].replace('.csv', '')
        
        # 'datetime' 컬럼을 최종 거래 시간으로 변환
        # (원래 날짜 정보에 end_time_str을 붙여서 정확한 datetime 객체 생성)
        df_temp['datetime'] = df_temp['datetime'].astype(str) + ' ' + end_time_str
        df_temp['datetime'] = pd.to_datetime(df_temp['datetime'], format='%Y%m%d %H%M', errors='coerce')
        
        # 긴 형태(Long format)로 변환
        df_temp_long = df_temp.melt(id_vars=['datetime'], var_name='symbol', value_name='close_price')
        df_minutely_list.append(df_temp_long.dropna(subset=['close_price']))
        
    except Exception as e:
        print(f"경고: 파일 {file_name} 처리 중 오류 발생: {e}")
        continue

if not df_minutely_list:
    print("오류: 분 단위 가격 데이터를 로드하지 못했습니다.")
    exit()

df_price_minutely = pd.concat(df_minutely_list, ignore_index=True)
df_price_minutely = df_price_minutely.sort_values(['symbol', 'datetime'])
print("✅ 분 단위 가격 데이터 통합 완료.")


# --- 2. 서프라이즈 데이터 병합 및 수익률 계산 ---
try:
    df_surprise = pd.read_csv(ARIMA_RESULT_PATH)
    df_surprise['date'] = pd.to_datetime(df_surprise['date'])
    df_surprise = df_surprise[['date', 'symbol', 'surprise_z']].dropna()
    
    # 시그널 타입 정의 (핵심 시그널: |Z| > 2)
    MEAN_Z_SCORE = df_surprise['surprise_z'].mean()
    STD_Z_SCORE = df_surprise['surprise_z'].std()
    
    df_surprise['signal_type'] = np.select(
        [
            df_surprise['surprise_z'] > MEAN_Z_SCORE + 2 * STD_Z_SCORE,
            df_surprise['surprise_z'] < MEAN_Z_SCORE - 2 * STD_Z_SCORE
        ],
        ['Positive_Signal', 'Negative_Signal'],
        default='No_Signal'
    )
    
    # 핵심 시그널만 추출
    df_signals = df_surprise[df_surprise['signal_type'] != 'No_Signal'].copy()

except FileNotFoundError:
    print("오류: ARIMA 결과 파일을 찾을 수 없습니다.")
    exit()

print(f"총 {len(df_signals)}개의 핵심 시그널에 대해 검증을 시작합니다.")

def calculate_pre_post_returns_minutely(row):
    """분 단위 데이터를 활용하여 공개 직전/직후 10분 수익률을 계산하는 함수"""
    sym = row['symbol']
    # 'date'는 수출 월의 마지막 날이므로, 발표일은 '다음 달 1일'로 설정
    # (공개일은 주말에도 데이터를 제공한다는 벤더 주장에 따라 1일로 가정)
    pub_date_day = row['date'] + pd.DateOffset(days=1)
    
    # 공개 시점 (T_pub): 다음 달 1일 10:20:00
    T_pub = pub_date_day.replace(hour=10, minute=20, second=0)
    
    # 직전 시점 (T_pre): 10:10:00 (10분 전)
    T_pre = pub_date_day.replace(hour=10, minute=10, second=0)
    
    # 직후 시점 (T_post): 10:30:00 (10분 후)
    T_post = pub_date_day.replace(hour=10, minute=30, second=0)

    # 해당 종목 가격 데이터 필터링
    df_sym = df_price_minutely[df_price_minutely['symbol'] == sym].set_index('datetime')
    
    # 1. 공개 전 가격 (Price Pre-T10): 10:10 종가
    P_pre = df_sym.loc[df_sym.index == T_pre, 'close_price'].values
    if len(P_pre) == 0: P_pre = np.nan
    else: P_pre = P_pre[0]

    # 2. 공개 시점 가격 (Price T_pub): 10:20 종가
    P_pub = df_sym.loc[df_sym.index == T_pub, 'close_price'].values
    if len(P_pub) == 0: P_pub = np.nan
    else: P_pub = P_pub[0]
    
    # 3. 공개 후 가격 (Price Post-T10): 10:30 종가
    P_post = df_sym.loc[df_sym.index == T_post, 'close_price'].values
    if len(P_post) == 0: P_post = np.nan
    else: P_post = P_post[0]
        
    # 수익률 계산:
    # R_pre: 10:10 ~ 10:20 (공개 직전 10분간의 시장 예상 반영 여부)
    R_pre = (P_pub / P_pre) - 1 if P_pub and P_pre and P_pre != 0 else np.nan
    
    # R_post: 10:20 ~ 10:30 (공개 직후 10분간의 반응)
    R_post = (P_post / P_pub) - 1 if P_post and P_pub and P_pub != 0 else np.nan

    return pd.Series([R_pre, R_post], index=['R_pre', 'R_post'])

# --- 3. 수익률 계산 적용 (tqdm 사용) ---
df_minutely_returns = df_signals.progress_apply(calculate_pre_post_returns_minutely, axis=1)
df_final_minutely = pd.concat([df_signals, df_minutely_returns], axis=1).dropna(subset=['R_pre', 'R_post'])
print("수익률 계산 완료.")


# --- 4. 최종 검증 요약 ---
df_summary_final = df_final_minutely.groupby('signal_type')[['R_pre', 'R_post']].mean().mul(100).round(4)
df_summary_final = df_summary_final.rename(columns={'R_pre': 'Avg_Return_Pre_10min (%)', 'R_post': 'Avg_Return_Post_10min (%)'})

print("\n" + "="*70)
print("과제 2 최종 검증: 벤더 데이터 공개 직전(R_pre) vs 직후(R_post) 반응")
print("="*70)
print(df_summary_final.to_markdown())
print("="*70)

➡️ 1. 분 단위 가격 데이터 통합 로드 시작...
✅ 분 단위 가격 데이터 통합 완료.
총 1176개의 핵심 시그널에 대해 검증을 시작합니다.


  3%|▎         | 35/1176 [00:17<09:58,  1.91it/s]

제공해주신 결과는 벤더의 주장이 거짓이라는 강력한 증거를 제시하며, 데이터가 시장 가격에 반영되기 전에 고객에게 도달하지 못했다는 것을 시사합니다.결론적으로, 이 데이터는 투자 가치가 크게 떨어집니다.📉 과제 2 최종 검증 결과 해석이 분석의 핵심은 $R_{\text{pre}} \approx 0$ (공개 직전 움직임 없음)이고 $R_{\text{post}}$ (공개 직후 움직임)이 시그널 방향과 일치해야 벤더 주장이 참이라는 것입니다.시그널 유형예상되는 Rpost​ 방향실제 Rpre​ (10:10~10:20)실제 Rpost​ (10:20~10:30)검증 결과Positive Signal상승 (양수)-0.0989%-0.0136%실패 (주가 방향이 반대)Negative Signal하락 (음수)-0.0594%+0.0287%실패 (주가 방향이 반대)1. 시그널 방향 예측 실패 (가장 큰 문제)Positive Signal이 발생하면 주가는 상승해야 하는데, 공개 직후 -0.0136% 하락했습니다.Negative Signal이 발생하면 주가는 하락해야 하는데, 공개 직후 +0.0287% 상승했습니다.해석: 이 시그널은 주가 방향을 예측하는 데 전혀 도움이 되지 않으며, 오히려 무작위이거나 예상과 반대로 움직였습니다. 이는 시그널의 근본적인 유효성에 문제가 있음을 보여줍니다.2. 시장 선반영 증거 ($R_{\text{pre}}$의 움직임)데이터가 공개된 10시 20분 직전 10분 동안($R_{\text{pre}}$) 이미 주가는 **-0.0989% ~ -0.0594%**의 비교적 큰 움직임을 보였습니다.반면, **공개 직후($R_{\text{post}}$)**의 움직임은 이에 비해 미미하거나 아예 반대로 움직였습니다.해석: 주가가 공식 데이터 발표 이전에 이미 정보를 입수하여 움직였을 가능성(정보 유출, 다른 경로를 통한 선행 파악 등)이 매우 높습니다. 벤더가 주장하는 '가장 빠른 데이터'는 이미 가격에 반영된 후에 도착한 것으로 판단됩니다.💡 최종 결론 (과제 2 보고)벤더의 **"가장 빠르게 제공한다"**는 주장은 거짓으로 판단됩니다.Surprise Z-Score 시그널은 주가 방향 예측에 실패했으며, 오히려 가격 움직임의 대부분은 데이터 공개 직전에 완료되었습니다.이 데이터는 주식 투자 시 단독 시그널로 사용하기에는 투자 가치가 크게 떨어진다고 보고해야 합니다.